# CopyNE: setup and baseline (Colab / Kaggle, T4)

Rebuilds the environment of CopyNE (ACL 2024), downloads the AISHELL-1 test audio and the released checkpoint,
and evaluates three decoding modes (CTC greedy, attention, CopyNE copy-attention) with CER and NE-CER.

**Status: not executed yet.** The scripts were written and checked statically (dependency resolution, data
preparation and metric scripts on dummy data); the GPU run and the checkpoint download still have to be done here.
Run the cells in order. Runtime: GPU (T4).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 0. Configuration
`CODE_DIR` must contain this folder (`code/copyne`) copied to Google Drive. `RESULTS_DIR` is where results are kept
across sessions. Colab sessions end, so everything that matters is written to Drive.

In [ ]:
import os
CODE_DIR    = '/content/drive/MyDrive/final_project/code/copyne'
RESULTS_DIR = '/content/drive/MyDrive/final_project/results/copyne'
WORK        = '/content/work'
CKPT_DIR    = '/content/ckpt'
CKPT_FOLDER_ID = '1KP6JD7vFNkeo9ZWKba0ToDpQzd4Z1lhT'   # released model, from the CopyNE README
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ.update(WORK=WORK)
REPO, VENV = f'{WORK}/CopyNE', f'{WORK}/venv310'

## 1. Environment
Clones CopyNE at a fixed commit and builds a Python 3.10 venv with the pinned packages
(`torch 2.0.1`, `torchaudio 2.0.2`, ...). A separate venv is used because the code needs torch 2.0.1 and `imp`,
which do not exist on the newer Python of the notebook kernel. Takes a few minutes.

In [ ]:
!bash {CODE_DIR}/setup_env.sh

## 2. Data
Downloads only the 20 AISHELL-1 test speakers (per-speaker tarballs from Hugging Face), fixes the manifest paths,
and writes the gold `.text` files. If Hugging Face is not reachable, use `--source openslr` (15 GB archive).

In [ ]:
!pip install -q huggingface_hub requests
!python {CODE_DIR}/prepare_data.py --repo {REPO}

## 3. Checkpoint
Downloads the released model folder. If Google Drive reports a download quota error, add the folder to your own
Drive (right click, *Add shortcut*), copy it, and set `CKPT_DIR` to the copy.

In [ ]:
!pip install -q gdown
!gdown --folder {CKPT_FOLDER_ID} -O {CKPT_DIR}
!find {CKPT_DIR} -type f | head -20

Look at the file list above. `run_demo.py` expects `avg.model` or `best.model` in one directory and picks the
vocabulary (`char.vocab` or `multi_char.vocab`) by matching the size of the checkpoint's output layer.
Set `MODEL_DIR` to that directory.

In [ ]:
MODEL_DIR = CKPT_DIR      # change if the files are in a sub-folder

## 4. Smoke test (50 utterances)

In [ ]:
!python {CODE_DIR}/run_demo.py --repo {REPO} --venv {VENV} --model-dir {MODEL_DIR} --limit 50 --out {WORK}/results/smoke

Check that `copy_attention` is lower than `ctc_greedy_search` on NE-CER and that the `.asr` files contain Chinese text.
If a mode crashes, read `<out>/<tag>.stdout` first.

## 5. Full run on `test-ne` (2244 utterances, 3082 entities)

In [ ]:
!python {CODE_DIR}/run_demo.py --repo {REPO} --venv {VENV} --model-dir {MODEL_DIR} --manifest test-ne --out {WORK}/results/test-ne

In [ ]:
import pandas as pd, shutil
df = pd.read_csv(f'{WORK}/results/test-ne/summary.csv')
display(df)
shutil.copytree(f'{WORK}/results/test-ne', f'{RESULTS_DIR}/test-ne', dirs_exist_ok=True)

## 6. Qualitative examples
Prints utterances where CopyNE fixes an entity that the baseline gets wrong.

In [ ]:
import json
def load(p): return dict(l.rstrip('\n').split(' ', 1) for l in open(p, encoding='utf8') if ' ' in l)
base = load(f'{WORK}/results/test-ne/test-ne.ctc_greedy_search.asr')
copy = load(f'{WORK}/results/test-ne/test-ne.copy_attention.asr')
gold = {}
nes = {}
for l in open(f'{REPO}/data_to_upload/aishell_dataset/test-ne.json', encoding='utf8'):
    d = json.loads(l); gold[d['key']] = d['txt']; nes[d['key']] = [d['txt'][a:b+1] for _, a, b in d['ne_lst']]
shown = 0
for k in gold:
    if k in base and k in copy and base[k] != gold[k] and copy[k] == gold[k]:
        print(k, '| entities:', nes[k]); print('  ref :', gold[k]); print('  base:', base[k]); print('  copy:', copy[k]); shown += 1
    if shown == 8: break